# Llama3.1-8B-Instruct Reinforcement Learning Demo

This notebook demonstrates training on Llama3.1-8B-Instruct model with either GRPO (Group Relative Policy Optimization) or GSPO (Group Sequence Policy Optimization).

This notebook can run on **TPU v6e-8** or **v5p-8**.

## What is GRPO/GSPO?

GRPO/GSPO is an RL algorithm that enhances reasoning abilities of LLMs by:
1. Generating multiple responses for each prompt
2. Evaluating responses using reward models  
3. Calculating relative advantages to update the policy

The difference is in the loss function - either it's optimizing each token (GRPO) or the whole sequence(GSPO).

## Prerequisites

### Change Runtime Type (only if running on Google Colab)

**Instructions:**
1.  Navigate to the menu at the top of the screen.
2.  Click on **Runtime**.
3.  Select **Change runtime type** from the dropdown menu.
4.  Select **v6e-8** or **v5p-8 TPU** as the **Hardware accelerator**.
5. Click on **Save**.

### Get Your Hugging Face Token

To access model checkpoint from the Hugging Face Hub, you need to authenticate with a personal access token.

**Follow these steps to get your token:**

1.  **Navigate to the Access Tokens page** in your Hugging Face account settings. You can go there directly by visiting this URL:
    *   [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

2.  **Create a new token** by clicking the **"+ Create new token"** button.

3.  **Give your token a name** and assign it a **`read` role**. The `read` role is sufficient for downloading models.

4.  **Copy the generated token**. You will need this in the later steps.

**Follow these steps to store your token (only if running on Google Colab):**

1. On the left sidebar of your Colab window, click the key icon (the Secrets tab).

2. Click **"+ Add new secret"**.

3. Set the Name as **HF_TOKEN**.

4. Paste your token into the Value field.

5. Ensure the Notebook access toggle is turned On.

In [ ]:
try:
  from google.colab import userdata
  print("Running the notebook on Google Colab")
  IN_COLAB = True
except ImportError:
    print("Running the notebook on Visual Studio or JupyterLab")
    IN_COLAB = False

## Installation: MaxText and Post training Dependencies

**Running the notebook on Visual Studio or JupyterLab:** Before proceeding, create a virtual environment and install the required post-training dependencies by following `Option 3: Installing [tpu-post-train]` in the [MaxText installation guide](https://maxtext.readthedocs.io/en/latest/install_maxtext.html#from-source). Once the environment is set up, ensure the notebook is running within it.

In [ ]:
if IN_COLAB:
    # Clone the MaxText repository
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext

    # Install uv, a fast Python package installer
    !pip install uv
    
    # Install MaxText and post-training dependencies
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps

**Session restart Instructions for Colab:**
1.  Navigate to the menu at the top of the screen.
2.  Click on **Runtime**.
3.  Select **Restart session** from the dropdown menu.

You will be asked to confirm the action in a pop-up dialog. Click on **Yes**.

## Environment Setup

In [ ]:
import datetime
import os
import sys
import subprocess
from pathlib import Path
from huggingface_hub import login
from etils import epath
import jax

from maxtext.trainers.post_train.rl.train_rl import rl_train
from maxtext.utils.model_creation_utils import setup_configs_and_devices
from maxtext.utils.globals import MAXTEXT_REPO_ROOT, MAXTEXT_PKG_DIR

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
os.environ["SKIP_JAX_PRECOMPILE"] = "1"  # Faster startup for vLLM
# Suppress vLLM logging with a severity level below ERROR
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"


print(f"MaxText installation path: {MAXTEXT_PKG_DIR}")

In [ ]:
if not jax.distributed.is_initialized():
  jax.distributed.initialize()
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

# If not found in the environment, prompt the user for input securely
# getpass function ensures the token is hidden while you type
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Hugging Face token not found in environment. Please enter it here: ")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("Authenticated with Hugging Face successfully!")
else:
    print("Authentication failed: Hugging Face token is not set.")

## Model Configurations

In [ ]:
MODEL_NAME = "llama3.1-8b-Instruct"
RUN_NAME = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
LOSS_ALGO="grpo" #  or "gspo-token" if you want to use GSPO

CHAT_TEMPLATE_PATH = f"{MAXTEXT_REPO_ROOT}/src/maxtext/examples/chat_templates/gsm8k_rl.json"
if not epath.Path(CHAT_TEMPLATE_PATH).exists():
    raise FileNotFoundError(f"Chat template not found: {CHAT_TEMPLATE_PATH}")

# set the path to the model checkpoint (excluding `/0/items`) or leave empty to download from HuggingFace
MODEL_CHECKPOINT_PATH = ""
if not MODEL_CHECKPOINT_PATH:
   MODEL_CHECKPOINT_PATH = f"{MAXTEXT_PKG_DIR}/llama_checkpoint"
   print("Model checkpoint will be downloaded from HuggingFace at: ",  MODEL_CHECKPOINT_PATH)
   print("Set MODEL_CHECKPOINT_PATH if you do not wish to download the checkpoint.")
    
OUTPUT_DIRECTORY = f"{MAXTEXT_PKG_DIR}/rl_llama3_output"

## Download Llama3.1-8B Model Checkpoint from Hugging Face

In [ ]:
if not epath.Path(MODEL_CHECKPOINT_PATH).exists():

    # Install torch for the conversion script
    print("Installing torch...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "torch", "--index-url", "https://download.pytorch.org/whl/cpu"
        ],
        check=True
    )

    # Run checkpoint conversion with environment variables
    print("Converting checkpoint from HuggingFace...")
    env = os.environ.copy()
    env["JAX_PLATFORMS"] = "cpu"

    subprocess.run(
        [
            sys.executable,
            "-m", "maxtext.checkpoint_conversion.to_maxtext",
            f"{MAXTEXT_PKG_DIR}/configs/base.yml",
            f"model_name={MODEL_NAME}",
            f"base_output_directory={MODEL_CHECKPOINT_PATH}",
            f"hf_access_token={HF_TOKEN}",
            "use_multimodal=false",
            "scan_layers=true",
            "skip_jax_distributed_system=True",
        ],
        check=True,
        env=env
    )

    if not epath.Path(MODEL_CHECKPOINT_PATH).exists():
        raise ValueError("Model checkpoint conversion failed. Check the logs above.")

## MaxText Configurations

In [ ]:
# Configuration for RL training
config_argv = [
    "",
    f"{MAXTEXT_PKG_DIR}/configs/post_train/rl.yml",
    f"model_name={MODEL_NAME}",
    f"run_name={RUN_NAME}",
    f"chat_template_path={CHAT_TEMPLATE_PATH}",
    f"load_parameters_path={MODEL_CHECKPOINT_PATH}/0/items",
    f"base_output_directory={OUTPUT_DIRECTORY}",
    f"hf_access_token={HF_TOKEN}",
    "debug.rl=False",
    f"rl.loss_algo={LOSS_ALGO}",
    "use_pathways=False",
    "log_config=False",
]

print("✓ Configuration initialized successfully")
print(f"📁 Output directory: {OUTPUT_DIRECTORY}")
print(f"🤖 Model: {MODEL_NAME}")

## RL Training

In [ ]:
import traceback

print("\n" + "=" * 80)
print(f"🚀 Starting {LOSS_ALGO} Training...")
print("=" * 80)
try:
    rl_train(argv=config_argv, kwargs={})
    print("\n" + "=" * 80)
    print("✅ Training Completed Successfully!")
    print("=" * 80)
except Exception:
    print("\n" + "=" * 80)
    print("❌Training Failed!")
    print("=" * 80)
    traceback.print_exc()
    sys.exit(1)

## 📚 Learn More

- **CLI Usage**: https://maxtext.readthedocs.io/en/latest/tutorials/rl.html
- **Configuration**: See `src/maxtext/configs/rl.yml` for all available options
- **Documentation**: Check `src/maxtext/trainers/post_train/rl/train_rl.py` for the `rl_train` function implementation